# Bitcoin Price Prediction — Exploratory Analysis Starter

Quick interactive exploration of the collected BTC-USD data and engineered
features. Run the pipeline first so the CSVs exist:

```bash
python scripts/download_data.py
python scripts/build_features.py
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(PROJECT_ROOT), str(PROJECT_ROOT / "src")]

import matplotlib.pyplot as plt
import pandas as pd

from src.btc_prediction.data.collector import load_processed
from src.btc_prediction.features.build_features import create_dataset

data = load_processed()
print(f"{len(data):,} daily rows, {data.index.min().date()} → {data.index.max().date()}")
data.tail()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
axes[0].plot(data.index, data["Close"], lw=0.8)
axes[0].set_title("BTC-USD daily close")
axes[0].grid(alpha=0.3)
returns = data["Close"].pct_change() * 100
axes[1].plot(returns.index, returns, lw=0.5, color="#F7931A")
axes[1].set_title("Daily returns (%)")
axes[1].grid(alpha=0.3)
plt.show()
print(returns.describe().round(2))

In [ ]:
# Feature dataset with leak-free targets
dataset = create_dataset(data, keep_helper_columns=True)
print(f"Feature dataset: {len(dataset):,} rows x {dataset.shape[1]} columns")

up_share = dataset["Direction"].mean()
print(f"UP-day share (classification balance): {up_share:.1%}")

corr = dataset[["Close", "RSI_14", "MACD", "Daily_Return_Pct",
                "Rolling_Volatility_30"]].corr()
corr.style.background_gradient(cmap="coolwarm", vmin=-1, vmax=1)

## Next steps

- Train models: `python scripts/train_models.py`
- Inspect genuine metrics: `reports/evaluation_summary.json`
- Launch the dashboard: `streamlit run app/app.py`